<a href="https://colab.research.google.com/github/henric00/PROJETO_IA/blob/main/Projeto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise e Modelagem Preditiva e Notas de Filmes (TMDB)
## Integrantes: Carlos Henrico Fontes Cabral
## Fonte de Dados: [Os 10.000 melhores filmes do TMDB](https://www.kaggle.com/datasets/siddharthbhakta/tmdb-top-10000-movies-updated-2026/data)


## Introdução

Este projeto utiliza um conjunto de dados disponibilizado na plataforma Kaggle, contendo informações sobre aproximadamente 10.000 filmes obtidas a partir da API do The Movie Database (TMDB). O dataset reúne diversas características dos filmes, como popularidade, avaliações dos usuários, quantidade de votos, data de lançamento, idioma original e outras informações relevantes para análise.

A escolha desse conjunto de dados deve-se à sua riqueza de atributos, que possibilita investigar relações entre diferentes características dos filmes e aplicar técnicas de Aprendizado de Máquina para construção de modelos preditivos. Além disso, por tratar-se de um tema amplamente conhecido, os resultados obtidos tornam-se mais intuitivos de interpretar e discutir ao longo do projeto.

# Obejtivo
O objetivo deste projeto é desenvolver um modelo de regressão capaz de estimar a nota média dos filmes `vote_average` a partir das características disponíveis no conjunto de dados. Além da construção do modelo preditivo, o estudo busca investigar a influência de características como popularidade, engajamento do público e informações temporais na avaliação média dos filmes, identificando quais atributos apresentam maior relação com a variável-alvo.

# Atributo-alvo
O atributo-alvo escolhido é o vote_average (`nota média`),
este campo representa a avaliação quantitativa atribuída pelos usuários do TMDB aos filmes. É a variável que buscamos estimar através do nosso modelo preditivo, servindo como o indicador de "sucesso" ou "qualidade" que pretendemos prever a partir de outras características.

# Atributos preditivos
Os atributos preditivos ou variáveis independentes são as características mensuráveis de cada filme utilizadas pelo modelo para aprender e realizar a estimativa da nota média `vote_average`. Inicialmente serão consideradas as principais características disponíveis no conjunto de dados. Após a análise exploratória e o pré-processamento, será definida a seleção final das variáveis utilizadas no treinamento. Selecionamos atributos que representam dimensões do engajamento e do contexto do filme: a `popularity` e o `vote_count` para capturar a percepção do público, o `release_year` para identificar tendências temporais e o original_language para considerar aspectos culturais.

# Tipo da tarefa: Regressão
Escolhemos a tarefa de regressão porque nosso objetivo é prever a nota média exata dos filmes (vote_average), que varia de forma contínua de 0 a 10. Não seria interessante usar classificação, pois isso nos obrigaria a criar categorias artificiais (como definir um limite para o que é "Bom" ou "Ruim"), o que faria o modelo perder a precisão do dado original. Com a regressão, mantemos a escala real da nota e conseguimos medir com mais exatidão o quanto o modelo acertou ou errou, permitindo uma análise muito mais detalhada da relação entre os atributos e a avaliação do público.

# Compreensão dos dados
O conjunto de dados utilizado possui 9.980 registros e 10 atributos. A base reúne informações detalhadas sobre filmes da plataforma TMDB, abrangendo dados de popularidade, identificação e avaliações dos usuários.

O conjunto de dados é composto por diferentes tipos de variáveis que desempenham papéis distintos no desenvolvimento do modelo de regressão. O nosso atributo-alvo principal é o vote_average, que representa a nota média atribuída pelos usuários do TMDB e é o valor que o modelo buscará estimar.

Para auxiliar nessa predição, utilizamos os atributos preditivos, que fornecem o contexto necessário para o aprendizado. As variáveis popularity e vote_count quantificam o engajamento e a base estatística de avaliação de cada obra. A variável release_date introduz o contexto temporal, permitindo que o modelo aprenda padrões relacionados à época de lançamento, enquanto original_language oferece uma dimensão sobre a origem cultural do filme.









In [25]:
# bibliotecas necessárias
import pandas as pd

# URL direta para o arquivo CSV no GitHub
url = "https://raw.githubusercontent.com/henric00/PROJETO_IA/refs/heads/main/data/moviesTMBD.csv"

# Carregando o dataset
df = pd.read_csv(url)

A base de dados apresenta una estrutura concisa, composta por:


*   Numéricas: popularity (float), vote_average (float) e vote_count (int).
*   Booleanas: adult (bool).
*   Textuais/Categóricas: title, original_language, overview e release_date (armazenadas como object).


In [26]:
# 1. Informações gerais sobre registros, atributos e tipos
print(f"Dimensões do Dataset: {df.shape[0]} linhas e {df.shape[1]} colunas.\n")
print("Tipos de dados:")
df.info()


Dimensões do Dataset: 9980 linhas e 10 colunas.

Tipos de dados:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9980 entries, 0 to 9979
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         9980 non-null   int64  
 1   id                 9980 non-null   int64  
 2   title              9980 non-null   object 
 3   popularity         9980 non-null   float64
 4   adult              9980 non-null   bool   
 5   original_language  9980 non-null   object 
 6   overview           9978 non-null   object 
 7   release_date       9978 non-null   object 
 8   vote_average       9980 non-null   float64
 9   vote_count         9980 non-null   int64  
dtypes: bool(1), float64(2), int64(3), object(4)
memory usage: 711.6+ KB


# Valores ausentes, Duplicações e Inconsistências


*  **Valores ausentes:** A base apresenta altíssima completude. Foram detectadas falhas de preenchimento apenas na coluna overview (2 registros) e na coluna release_date (2 registros).

*  **Duplicações:** A verificação por registros idênticos retornou 0 ocorrências, o que confirma a exclusividade das observações e elimina o risco do modelo aprender padrões enviesados por dados repetidos.



In [27]:
# 2. Verificando ausentes, duplicados e possíveis inconsistências
print("Valores ausentes por coluna:")
print(df.isnull().sum())

print(f"\nTotal de linhas completamente duplicadas: {df.duplicated().sum()}")

# Checando inconsistências (filmes com 0 votos que tenham notas)
filmes_zero_votos = (df['vote_count'] == 0).sum()
print(f"\nInconsistências: Filmes com 0 votos registrados = {filmes_zero_votos}")

Valores ausentes por coluna:
Unnamed: 0           0
id                   0
title                0
popularity           0
adult                0
original_language    0
overview             2
release_date         2
vote_average         0
vote_count           0
dtype: int64

Total de linhas completamente duplicadas: 0

Inconsistências: Filmes com 0 votos registrados = 0


# Distribuição do atributo-alvo e Desbalanceamento


*  Distribuição do alvo (vote_average): O atributo-alvo apresenta média de 6,75 e mediana de 6,71, com desvio padrão baixo (0,64). As notas da base variam em um intervalo estreito de 5,5 a 8,98.
*  analisando a distribuição contínua, observamos uma forte concentração (enviesamento). A ausência absoluta de notas abaixo de 5,5 indica que este dataset não contém filmes considerados "fracassos" totais, concentrando as produções em faixas de avaliação medianas e altas. Essa concentração precisará ser considerada na avaliação do modelo, pois ele terá menos exemplos para aprender a prever notas extremas.





In [28]:
# 3. Estatísticas descritivas do atributo-alvo
print("Estatísticas da Nota Média (vote_average):")
print(df['vote_average'].describe())

Estatísticas da Nota Média (vote_average):
count    9980.000000
mean        6.750276
std         0.641034
min         5.500000
25%         6.249750
50%         6.716500
75%         7.216000
max         8.987000
Name: vote_average, dtype: float64
